# sklearn regression workflow

Every step on a tiny dataset first, so you can check the numbers by eye. The real hourly
power data comes at the end, using exactly the same calls.

**What's in here**
- fit a `LinearRegression`, read `coef_` and `intercept_`
- `predict`, residuals, RMSE / MAE / R² / MAPE computed by hand and by sklearn
- two features, coefficient table
- chronological split vs `train_test_split(shuffle=True)`
- `Ridge`, `Lasso`, `ElasticNet`: what alpha does to the coefficients
- hour as a number vs one-hot; interactions; `PolynomialFeatures`
- baselines the model must beat
- residual diagnostics by group
- the same workflow on the real data
- what to say when R² looks too good

In [1]:
import numpy as np
import pandas as pd

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 30)
np.set_printoptions(precision=4, suppress=True)

## 1. One feature, exact line

Five rows. `y` is exactly `2 * x1 + 3`, so we know the answer the model should find.

In [2]:
X = pd.DataFrame({"x1": [1, 2, 3, 4, 5]})
y = pd.Series([5, 7, 9, 11, 13], name="y")      # 2*x1 + 3
pd.concat([X, y], axis=1)

,x1,y
0,1,5
1,2,7
2,3,9
3,4,11
4,5,13


In [3]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X, y)
print("coef_      :", model.coef_)
print("intercept_ :", model.intercept_)

coef_      : [2.]
intercept_ : 3.0


`coef_` is 2 (the slope on x1) and `intercept_` is 3, as built. `coef_` is an array with
one entry per column of X.

In [4]:
pred = model.predict(X)
pd.DataFrame({"x1": X["x1"], "y": y, "pred": pred})

,x1,y,pred
0,1,5,5.0
1,2,7,7.0
2,3,9,9.0
3,4,11,11.0
4,5,13,13.0


## 2. Metrics by hand, then by sklearn

Now `y` has a bit of noise so the residuals are not all zero.

In [5]:
y = pd.Series([5, 8, 8, 11, 14], name="y")      # 2*x1 + 3 plus noise
model = LinearRegression().fit(X, y)
pred = model.predict(X)
tbl = pd.DataFrame({"y": y, "pred": pred.round(2)})
tbl["error"] = tbl["y"] - tbl["pred"]
tbl

,y,pred,error
0,5,5.0,0.0
1,8,7.1,0.9
2,8,9.2,-1.2
3,11,11.3,-0.3
4,14,13.4,0.6


In [6]:
print("mean squared error :", (tbl["error"] ** 2).mean())
print("RMSE               :", np.sqrt((tbl["error"] ** 2).mean()))
print("MAE                :", tbl["error"].abs().mean())

mean squared error : 0.5399999999999998
RMSE               : 0.7348469228349533
MAE                : 0.6


In [7]:
from sklearn.metrics import mean_squared_error, root_mean_squared_error, mean_absolute_error, r2_score

print("mean_squared_error      :", mean_squared_error(y, pred))
print("root_mean_squared_error :", root_mean_squared_error(y, pred))
print("mean_absolute_error     :", mean_absolute_error(y, pred))

mean_squared_error      : 0.54
root_mean_squared_error : 0.7348469228349535
mean_absolute_error     : 0.6


Same numbers. R² compares the model's squared error to the squared error of "always predict the mean".

In [8]:
ss_res = (tbl["error"] ** 2).sum()
ss_tot = ((y - y.mean()) ** 2).sum()
print("ss_res :", ss_res)
print("ss_tot :", ss_tot)
print("R2 by hand :", 1 - ss_res / ss_tot)
print("r2_score   :", r2_score(y, pred))

ss_res : 2.699999999999999
ss_tot : 46.8
R2 by hand : 0.9423076923076923
r2_score   : 0.9423076923076923


**Pitfall:** `r2_score(y_true, y_pred)` in that order. Swapping the arguments gives a different number.

In [9]:
print("r2_score(y, pred):", round(r2_score(y, pred), 4))
print("r2_score(pred, y):", round(r2_score(pred, y), 4), " <- wrong order, different number")

r2_score(y, pred): 0.9423
r2_score(pred, y): 0.9388  <- wrong order, different number


MAPE divides each error by the true value. One true value near zero makes it explode.

In [10]:
from sklearn.metrics import mean_absolute_percentage_error

y_small = pd.Series([100, 100, 0.1])
p_small = pd.Series([101, 99, 1.1])
print("errors     :", (y_small - p_small).values)
print("|err|/|y|  :", ((y_small - p_small).abs() / y_small.abs()).values)
print("MAPE       :", mean_absolute_percentage_error(y_small, p_small))

errors     : [-1.  1. -1.]
|err|/|y|  : [ 0.01  0.01 10.  ]
MAPE       : 3.34


Two errors of 1% and one error of 1 unit on a true value of 0.1 give a MAPE of 3.34 (334%).
Never use MAPE on prices (they cross zero).

## 3. Two features and a coefficient table

`y = 2*x1 + 3*x2 + 1`. Put the coefficients in a Series with the column names, so you can read them.

In [11]:
X = pd.DataFrame({"x1": [1, 2, 3, 4, 5, 6],
                  "x2": [1, 0, 2, 1, 3, 2]})
y = 2 * X["x1"] + 3 * X["x2"] + 1
pd.concat([X, y.rename("y")], axis=1)

,x1,x2,y
0,1,1,6
1,2,0,5
2,3,2,13
3,4,1,12
4,5,3,20
5,6,2,19


In [12]:
model = LinearRegression().fit(X, y)
coefs = pd.Series(model.coef_, index=X.columns)
print(coefs)
print("intercept:", model.intercept_)

x1    2.0
x2    3.0
dtype: float64
intercept: 1.0000000000000018


## 4. Chronological split vs shuffled split

Six rows with a date. For time series we train on the past and test on the future.

In [13]:
ts = pd.DataFrame({"date": pd.date_range("2023-01-01", periods=6, freq="D"),
                   "x": [1, 2, 3, 4, 5, 6],
                   "y": [10, 12, 14, 16, 18, 20]})
ts

,date,x,y
0,2023-01-01,1,10
1,2023-01-02,2,12
2,2023-01-03,3,14
3,2023-01-04,4,16
4,2023-01-05,5,18
5,2023-01-06,6,20


In [14]:
train = ts.iloc[:4]
test = ts.iloc[4:]
print("train"); print(train)
print()
print("test"); print(test)

train
        date  x   y
0 2023-01-01  1  10
1 2023-01-02  2  12
2 2023-01-03  3  14
3 2023-01-04  4  16

test
        date  x   y
4 2023-01-05  5  18
5 2023-01-06  6  20


Train is the first four days, test the last two. Now the same six rows through `train_test_split`, which shuffles by default.

In [15]:
from sklearn.model_selection import train_test_split

tr, te = train_test_split(ts, test_size=2, random_state=0)
print("train"); print(tr.sort_index())
print()
print("test"); print(te.sort_index())

train
        date  x   y
0 2023-01-01  1  10
1 2023-01-02  2  12
3 2023-01-04  4  16
4 2023-01-05  5  18

test
        date  x   y
2 2023-01-03  3  14
5 2023-01-06  6  20


The test days sit between training days. With hourly data, the hour before and the hour after
a test row are in training and look almost identical to it, so the score measures interpolation,
not forecasting.

**Interview check:** "Why not shuffle?" Because adjacent hours are near-copies; a shuffled
split leaks the neighbours. Always split by time (and see notebook 03 for `TimeSeriesSplit`).

## 5. Ridge, Lasso, ElasticNet: what alpha does

`Ridge` adds a penalty on the size of the coefficients. Larger alpha, smaller coefficients.
Same toy as section 3.

In [16]:
from sklearn.linear_model import Ridge

for alpha in [0, 1, 10, 100]:
    r = Ridge(alpha=alpha).fit(X, y)
    print(f"alpha={alpha:>4}  coef x1={r.coef_[0]:.3f}  coef x2={r.coef_[1]:.3f}  intercept={r.intercept_:.3f}")

alpha=   0  coef x1=2.000  coef x2=3.000  intercept=1.000
alpha=   1  coef x1=2.083  coef x2=2.455  intercept=1.526
alpha=  10  coef x1=1.701  coef x2=1.190  intercept=4.763
alpha= 100  coef x1=0.450  coef x2=0.252  intercept=10.548


alpha=0 is plain OLS (2, 3). As alpha grows both coefficients shrink toward 0 and the intercept absorbs the mean.

`Lasso` can set a coefficient to exactly zero. Add a useless third column that is pure noise.

In [17]:
from sklearn.linear_model import Lasso

X3 = X.assign(noise=[0.1, -0.2, 0.05, 0.3, -0.1, 0.2])
for alpha in [0.01, 0.1, 0.5]:
    l = Lasso(alpha=alpha).fit(X3, y)
    print(f"alpha={alpha:<5} coefs:", dict(zip(X3.columns, l.coef_.round(3))))

alpha=0.01  coefs: {'x1': 2.001, 'x2': 2.988, 'noise': 0.0}
alpha=0.1   coefs: {'x1': 2.011, 'x2': 2.878, 'noise': 0.0}
alpha=0.5   coefs: {'x1': 2.056, 'x2': 2.389, 'noise': 0.0}


The `noise` coefficient is exactly 0 already at alpha=0.01; at 0.5 the useful ones shrink too.
`ElasticNet` mixes the two penalties (`l1_ratio` between 0 = Ridge and 1 = Lasso).

In [18]:
from sklearn.linear_model import ElasticNet

e = ElasticNet(alpha=0.1, l1_ratio=0.5).fit(X3, y)
dict(zip(X3.columns, e.coef_.round(3)))

{'x1': 2.042, 'x2': 2.746, 'noise': -0.0}

## 6. Hour as a number vs one-hot

Four hours (0..3) and a `y` that peaks at hour 2. A straight line through hour cannot
bend, so it fits badly.

In [19]:
h = pd.DataFrame({"hour": [0, 1, 2, 3]})
y_h = pd.Series([1, 3, 5, 2])
lin = LinearRegression().fit(h, y_h)
pd.DataFrame({"hour": h["hour"], "y": y_h, "pred_numeric": lin.predict(h).round(2)})

,hour,y,pred_numeric
0,0,1,2.0
1,1,3,2.5
2,2,5,3.0
3,3,2,3.5


`pd.get_dummies` turns the single column into one True/False column per hour.

In [20]:
h_onehot = pd.get_dummies(h["hour"], prefix="hour")
h_onehot

,hour_0,hour_1,hour_2,hour_3
0,True,False,False,False
1,False,True,False,False
2,False,False,True,False
3,False,False,False,True


In [21]:
oh = LinearRegression().fit(h_onehot, y_h)
pd.DataFrame({"hour": h["hour"], "y": y_h, "pred_onehot": oh.predict(h_onehot).round(2)})

,hour,y,pred_onehot
0,0,1,1.0
1,1,3,3.0
2,2,5,5.0
3,3,2,2.0


With one-hot each hour gets its own level, so the fit is exact. `drop_first=True` drops one
column to avoid redundancy with the intercept (a coefficient per hour plus an intercept is one
more parameter than needed).

In [22]:
pd.get_dummies(h["hour"], prefix="hour", drop_first=True)

,hour_1,hour_2,hour_3
0,False,False,False
1,True,False,False
2,False,True,False
3,False,False,True


## 7. Interactions and PolynomialFeatures

An interaction is just a new column that multiplies two others. Build it by hand.

In [23]:
Xi = pd.DataFrame({"x1": [1, 2, 3], "x2": [0, 1, 1]})
Xi["x1_x2"] = Xi["x1"] * Xi["x2"]
Xi

,x1,x2,x1_x2
0,1,0,0
1,2,1,2
2,3,1,3


`PolynomialFeatures(degree=2)` builds all squares and products for you.

In [24]:
from sklearn.preprocessing import PolynomialFeatures

pf = PolynomialFeatures(degree=2, include_bias=False)
arr = pf.fit_transform(Xi[["x1", "x2"]])
pd.DataFrame(arr, columns=pf.get_feature_names_out())

,x1,x2,x1^2,x1 x2,x2^2
0,1.0,0.0,1.0,0.0,0.0
1,2.0,1.0,4.0,2.0,1.0
2,3.0,1.0,9.0,3.0,1.0


## 8. Baselines the model must beat

`DummyRegressor` predicts the training mean. A "naive" forecast predicts the previous value.

In [25]:
from sklearn.dummy import DummyRegressor

d = DummyRegressor(strategy="mean").fit(X, y)
print("predicts:", d.predict(X))
print("train mean:", y.mean())

predicts: [12.5 12.5 12.5 12.5 12.5 12.5]
train mean: 12.5


In [26]:
series = pd.Series([10, 12, 11, 15, 14, 16], name="y")
naive = series.shift(1)                # yesterday's value as today's forecast
pd.DataFrame({"y": series, "naive": naive, "error": series - naive})

,y,naive,error
0,10,NaN,NaN
1,12,10.0,2.0
2,11,12.0,-1.0
3,15,11.0,4.0
4,14,15.0,-1.0
5,16,14.0,2.0


In [27]:
print("naive RMSE:", root_mean_squared_error(series[1:], naive[1:]))

naive RMSE: 2.280350850198276


For hourly data the natural naive forecasts are `shift(24)` (same hour yesterday) and `shift(168)` (same hour last week).
If a model does not beat them, it has learned nothing.

## 9. Residual diagnostics by group

Six rows with an `hour` column. The mean error per hour shows where the model is biased.

In [28]:
res = pd.DataFrame({"hour": [0, 1, 0, 1, 0, 1],
                    "y":    [10, 20, 11, 22, 12, 24],
                    "pred": [10, 18, 11, 19, 12, 21]})
res["error"] = res["y"] - res["pred"]
res

,hour,y,pred,error
0,0,10,10,0
1,1,20,18,2
2,0,11,11,0
3,1,22,19,3
4,0,12,12,0
5,1,24,21,3


In [29]:
res.groupby("hour")["error"].mean()

hour
0    0.000000
1    2.666667
Name: error, dtype: float64

Hour 0 is unbiased, hour 1 is under-predicted by 2.7 on average. The overall mean error
(1.33) hides that. Always look at errors by hour, weekday and month.

## 10. The same workflow on the real data

In [30]:
df = pd.read_csv("../data/hourly_power_clean.csv", parse_dates=["time"]).sort_values("time").reset_index(drop=True)
df["hour"] = df["time"].dt.hour
df["dow"] = df["time"].dt.dayofweek
df["month"] = df["time"].dt.month
df["weekend"] = (df["dow"] >= 5).astype(int)
df["hdd"] = np.clip(15 - df["temp_c"], 0, None)     # heating degrees: how far below 15C
df["cdd"] = np.clip(df["temp_c"] - 22, 0, None)     # cooling degrees: how far above 22C
print(df.shape)
df[["time", "consumption_mwh", "temp_c", "hour", "dow", "weekend", "hdd", "cdd"]].head(3)

(17520, 12)


,time,consumption_mwh,temp_c,hour,dow,weekend,hdd,cdd
0,2022-01-01 00:00:00+00:00,26858.4,0.11,0,5,1,14.89,0.0
1,2022-01-01 01:00:00+00:00,26177.8,-0.18,1,5,1,15.18,0.0
2,2022-01-01 02:00:00+00:00,26229.4,-1.11,2,5,1,16.11,0.0


In [31]:
target = "consumption_mwh"
features = ["temp_c", "hdd", "cdd", "wind_ms", "solar_wm2", "hour", "weekend"]

split = int(len(df) * 0.8)                     # first 80% of the rows = earliest 80% of time
train = df.iloc[:split]
test = df.iloc[split:]
print("train:", train["time"].min().date(), "->", train["time"].max().date(), len(train), "rows")
print("test :", test["time"].min().date(), "->", test["time"].max().date(), len(test), "rows")

train: 2022-01-01 -> 2023-08-07 14016 rows
test : 2023-08-08 -> 2023-12-31 3504 rows


In [32]:
model = LinearRegression().fit(train[features], train[target])
pred = model.predict(test[features])
print("RMSE :", round(root_mean_squared_error(test[target], pred), 1))
print("MAE  :", round(mean_absolute_error(test[target], pred), 1))
print("R2   :", round(r2_score(test[target], pred), 3))

RMSE : 2953.6
MAE  : 2413.7
R2   : 0.436


In [33]:
pd.Series(model.coef_, index=features).round(2)

temp_c         39.75
hdd           429.19
cdd            49.35
wind_ms        17.05
solar_wm2       7.92
hour          357.55
weekend     -2217.57
dtype: float64

`hour` as a number gets one slope. Replace it with one-hot columns and compare.

In [34]:
X_train_oh = pd.get_dummies(train[features], columns=["hour"], drop_first=True)
X_test_oh = pd.get_dummies(test[features], columns=["hour"], drop_first=True)
print(X_train_oh.shape, "columns now")
model_oh = LinearRegression().fit(X_train_oh, train[target])
pred_oh = model_oh.predict(X_test_oh)
print("RMSE hour numeric :", round(root_mean_squared_error(test[target], pred), 1))
print("RMSE hour one-hot :", round(root_mean_squared_error(test[target], pred_oh), 1))

(14016, 29) columns now
RMSE hour numeric : 2953.6
RMSE hour one-hot : 772.9


Baselines: same hour yesterday and same hour last week, computed on the whole frame with `shift`
and then evaluated on the test rows only.

In [35]:
naive24 = df[target].shift(24).iloc[split:]
naive168 = df[target].shift(168).iloc[split:]
print("RMSE same hour yesterday  :", round(root_mean_squared_error(test[target], naive24), 1))
print("RMSE same hour last week  :", round(root_mean_squared_error(test[target], naive168), 1))
print("RMSE model (one-hot hour) :", round(root_mean_squared_error(test[target], pred_oh), 1))

RMSE same hour yesterday  : 1622.9
RMSE same hour last week  : 1418.5
RMSE model (one-hot hour) : 772.9


The model beats both naive forecasts. Now regularised models and a non-linear one, in one table.

In [36]:
from sklearn.ensemble import RandomForestRegressor

rows = []
for name, m in [("OLS", LinearRegression()),
                ("Ridge(1)", Ridge(alpha=1.0)),
                ("Lasso(1)", Lasso(alpha=1.0, max_iter=5000)),
                ("RandomForest", RandomForestRegressor(n_estimators=30, max_depth=8, random_state=0))]:
    m.fit(X_train_oh, train[target])
    p = m.predict(X_test_oh)
    rows.append({"model": name,
                 "rmse": round(root_mean_squared_error(test[target], p), 1),
                 "r2": round(r2_score(test[target], p), 3)})
pd.DataFrame(rows)

,model,rmse,r2
0,OLS,772.9,0.961
1,Ridge(1),774.0,0.961
2,Lasso(1),776.2,0.961
3,RandomForest,1287.8,0.893


In [37]:
out = test[["time", target]].copy()
out["pred"] = pred_oh
out["error"] = out[target] - out["pred"]
out["error"].describe().round(1)

count    3504.0
mean     -132.6
std       761.5
min     -2989.8
25%      -651.0
50%      -138.2
75%       386.1
max      2924.1
Name: error, dtype: float64

In [38]:
out.groupby(out["time"].dt.hour)["error"].mean().round(0)

time
0    -245.0
1    -188.0
2    -136.0
3    -171.0
4    -202.0
5    -140.0
6    -160.0
7     -85.0
8     -97.0
9     -84.0
10   -131.0
11   -108.0
12   -104.0
13    -64.0
14    -71.0
15   -132.0
16    -93.0
17    -93.0
18   -127.0
19    -69.0
20    -96.0
21    -86.0
22   -239.0
23   -259.0
Name: error, dtype: float64

Every hour is over-predicted on average (all errors negative), worst around 22:00-01:00.
The test period is late 2023, when demand is lower than in training (a slow downward trend the
features do not know about). A level shift plus a time-of-day pattern is the first thing to
say out loud after seeing the R².

## What to say when R² looks too good

1. Which feature has the largest coefficient, and is it derived from the target?
2. Is the split chronological, and does the test period come after training?
3. Does the model beat `shift(24)` and `shift(168)` by a believable margin?
4. Train R² vs test R²: a big gap is overfitting, an equal and very high pair is often leakage.
5. Is the target shifted the right way (negative shift = future)?
6. Any duplicated timestamps in the test set?

## Quick reference

| Task | Call |
|---|---|
| fit / predict | `m = LinearRegression().fit(X, y)`; `m.predict(X_new)` |
| coefficients with names | `pd.Series(m.coef_, index=X.columns)` |
| metrics | `root_mean_squared_error`, `mean_absolute_error`, `r2_score(y_true, y_pred)` |
| chronological split | `train = df.iloc[:n]`, `test = df.iloc[n:]` |
| shrinkage | `Ridge(alpha=)`, `Lasso(alpha=)`, `ElasticNet(alpha=, l1_ratio=)` |
| one-hot | `pd.get_dummies(df, columns=["hour"], drop_first=True)` |
| polynomial terms | `PolynomialFeatures(degree=2, include_bias=False)` |
| mean baseline | `DummyRegressor(strategy="mean")` |
| naive baseline | `y.shift(24)`, `y.shift(168)` |
| error by group | `out.groupby(out.time.dt.hour)["error"].mean()` |